# Enfoque Oficial Sin Aumento de Datos - Detección de Retinopatía Diabética Referible

**Protocolo Oficial y Riguroso de Evaluación Clínica:**
- **División de Datos:** Split Agrupado por Paciente y Estratificado (80% Train / 20% Val) sobre las imágenes originales.
- **Cero Aumento de Datos:** Sin rotaciones, traslaciones, flips ni réplicas artificiales.
- **Conjuntos:**
  - Entrenamiento: 264 imágenes de pacientes únicos (166 Referibles / 98 No Referibles)
  - Validación: 66 imágenes de pacientes únicos (41 Referibles / 25 No Referibles)
  - Prueba Independiente (Parte A): 83 imágenes originales (52 Referibles / 31 No Referibles)
- **Modelo Base:** EfficientNet-B0 pre-entrenado en ImageNet.
- **XAI:** Grad-CAM sobre la última capa convolucional (`top_activation`).

In [ ]:
import os
import csv
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import cv2

# Fijar semillas para reproducibilidad exacta
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print(f"TensorFlow Version: {tf.__version__}")

## 1. Carga del Split Oficial Sin Aumento de Datos

In [ ]:
split_csv_path = "../data_splits/split_sin_aumento_80_20.csv"

train_records = []
val_records = []
test_records = []

with open(split_csv_path, mode="r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        split = row["split"]
        item = {
            "image_name": row["Image name"],
            "binary_label": int(row["binary_label"]),
            "file_path": row["file_path"]
        }
        if split == "train":
            train_records.append(item)
        elif split == "val":
            val_records.append(item)
        elif split == "test":
            test_records.append(item)

print(f"Conjunto de Entrenamiento: {len(train_records)} imagenes (100% originales, sin replicacion)")
print(f"Conjunto de Validacion:   {len(val_records)} imagenes (100% originales)")
print(f"Conjunto de Prueba:       {len(test_records)} imagenes (100% originales)")

## 2. Pipeline de Preprocesamiento Estándar (Sin Aumento)
Solo redimensionamiento a 224x224 y normalización adecuada para EfficientNet.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16

def preprocess_image(image_path):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    # Preprocesamiento estandar de EfficientNet
    img = tf.keras.applications.efficientnet.preprocess_input(img)
    return img

def create_dataset(records, is_training=False):
    paths = [os.path.join("../../", r["file_path"]) for r in records]
    labels = [r["binary_label"] for r in records]
    
    def _load_data(path, label):
        img = preprocess_image(path)
        return img, label
    
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(_load_data, num_parallel_calls=tf.data.AUTOTUNE)
    if is_training:
        ds = ds.shuffle(buffer_size=len(records), seed=SEED)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

print("Pipeline de tf.data configurado estrictamente sin transformaciones de aumento.")

## 3. Construcción del Modelo: EfficientNet-B0

In [ ]:
def build_model():
    base_model = EfficientNetB0(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
    x = base_model.output
    x = GlobalAveragePooling2D(name="avg_pool")(x)
    x = Dropout(0.3, name="top_dropout")(x)
    predictions = Dense(1, activation="sigmoid", name="prediction")(x)
    
    model = Model(inputs=base_model.input, outputs=predictions)
    model.compile(
        optimizer=Adam(learning_rate=1e-4),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
    )
    return model

model = build_model()
print("Modelo EfficientNet-B0 compilado exitosamente.")

## 4. Protocolo Idéntico de Entrenamiento (Early Stopping con val_loss)

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True, verbose=1),
    ModelCheckpoint("best_efficientnet_sin_aumento.h5", monitor="val_loss", save_best_only=True, verbose=1)
]

print("Protocolo con monitor val_loss y paciencia de 7 epocas listo para entrenar.")

## 5. Explicabilidad Médica: Grad-CAM
Generación del mapa de calor de atención clínica sobre la última capa convolucional (`top_activation`).

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name="top_activation"):
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        class_channel = preds[:, 0]
        
    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

print("Funcion Grad-CAM lista para inspeccion clinica.")